# HVIA Data Analysis Training Task  
## Brazilian E-Commerce Public Dataset by Olist

- **Intern:** Ahmed Akram Amer
- **Date:** September 2026  
- **Task:** Training_Task – Olist Business Analysis & HVIA Solution Proposal

## 1. Introduction & Objective

This notebook presents a complete analysis of the Olist Brazilian E-Commerce dataset as part of the HVIA Data Analysis Training Task.

**Objectives:**
- Understand Olist’s business model and ecosystem
- Explore and understand the structure of the provided datasets
- Identify key business patterns, problems, and opportunities
- Interpret findings from a business perspective
- Propose realistic Data / AI solutions that HVIA could offer Olist
- Draft a professional outreach message to Olist decision-makers

## 2. Company Research – Understanding Olist

### 2.1 What is Olist?
Olist is the largest department store in Brazilian marketplaces. It operates as a marketplace platform that connects small and medium-sized businesses (sellers) from all over Brazil to major e-commerce channels(Amazon, Mercado Livre, ...etc.) through a single contract.

### 2.2 Business Model
- **For Sellers:** Olist helps small merchants sell their products on multiple marketplaces without managing multiple contracts. Sellers ship products using Olist’s logistics partners.
- **For Customers:** Customers buy from the “Olist Store” on various marketplaces and receive products from different sellers.
- **Revenue Streams:** Commission on sales, logistics fees, and value-added services.

### 2.3 Key Stakeholders
- Small & medium sellers (merchants)
- End customers
- Logistics partners
- Marketplace channels

### 2.4 Challenges Olist Likely Faces
- Delivery performance and late shipments
- Customer satisfaction and review management
- Seller quality control
- High competition in Brazilian e-commerce
- Logistics complexity across a large country

### 2.5 Why Data & AI Matter for Olist
Data can help Olist improve delivery times, predict customer dissatisfaction, optimize seller performance, and increase retention.

## 3. Dataset Understanding

### 3.1 Overview of the Dataset
The dataset contains real commercial data of ~100,000 orders from 2016 to 2018 made at Olist Store across multiple Brazilian marketplaces.

**Main Files:**
| File | Description |
|------|-------------|
| olist_customers_dataset.csv | Customer information and location |
| olist_geolocation_dataset.csv | Brazilian zip codes with lat/lng |
| olist_orders_dataset.csv | Order status and timestamps |
| olist_order_items_dataset.csv | Items inside each order + seller + price |
| olist_order_payments_dataset.csv | Payment methods and values |
| olist_order_reviews_dataset.csv | Customer reviews and scores |
| olist_products_dataset.csv | Product attributes and categories |
| olist_sellers_dataset.csv | Seller location information |
| product_category_name_translation.csv | Category name translation (PT → EN) |

### 3.2 Key Relationships
- One Order → One Customer
- One Order → Multiple Items (possible)
- One Item → One Seller
- One Order → One or more Payments
- One Order → One Review (usually)

### 3.3 Important Notes
- `customer_id` is unique per order (not per person)
- `customer_unique_id` identifies the same customer across orders
- An order can contain products from different sellers
- All company names in reviews were anonymized (Game of Thrones houses)

## 4. Data Loading & Preparation

### 4.1 Install Dependencies

In [1]:
# !pip install pandas kagglehub[pandas-datasets]

### 4.2 Import Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
import kagglehub
from kagglehub import KaggleDatasetAdapter

### 4.3 Load all datasets

In [3]:
customers = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_customers_dataset.csv'
)

geolocation = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_geolocation_dataset.csv'
)

orders = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_orders_dataset.csv'
)

order_items = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_order_items_dataset.csv'
)

order_payments = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_order_payments_dataset.csv'
)

order_reviews = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_order_reviews_dataset.csv'
)

products = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_products_dataset.csv'
)

sellers = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'olist_sellers_dataset.csv'
)

category_translation = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "olistbr/brazilian-ecommerce",
    'product_category_name_translation.csv'
)

In [4]:
# Quick look ath the shapes
datasets = {
    'customers': customers,
    'sellers': sellers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
    'order_reviews': order_reviews,
    'order_payments': order_payments,
    'category_translation': category_translation
}

for name, dataset in datasets.items():
    print(f"{name} - Dataset")
    print(f"    | Shape = {dataset.shape}")
    print(f"    | First 5 Rows \n{dataset.head()}")
    print("____________"*10)

customers - Dataset
    | Shape = (99441, 5)
    | First 5 Rows 
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   
3  b2b6027bc5c5109e529d4dc6358b12c3  259dac757896d24d7702b9acbbff3f3c   
4  4f2d8ab171c80ec8364f7c12e35b23ad  345ecd01c38d18a9036ed96c73b8d066   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  
3                      8775        mogi das cruzes             SP  
4                     13056               campinas             SP  
________________________________________________________________________________________

## 5. Exploratory Data Analysis
- Change OverTime
- Cumulative Analysis
- Performance Analysis
- Data Segmentation
- Part-of-Whole Analysis

### 5.1 Sales Performance Overtime

In [5]:
orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

#### Fix Data Types

In [6]:
orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


In [7]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [30]:
orders['order_delivered_carrier_date'] = pd.to_datetime(orders['order_delivered_carrier_date'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])
orders['order_estimated_delivery_date'] = pd.to_datetime(orders['order_estimated_delivery_date'])
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_approved_at'] = pd.to_datetime(orders['order_approved_at'])

# Extract Year & Month
orders['order_year'] = orders['order_purchase_timestamp'].dt.year
orders['order_month'] = orders['order_purchase_timestamp'].dt.month
orders['order_year_month'] = orders['order_purchase_timestamp'].dt.to_period("M")

In [31]:
# Join Order Items with Orders to get Sales with Dates

sales = order_items.merge(
    orders,
    on='order_id',
    how='inner'
)

sales.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,-9.0,2017,9,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,-3.0,2017,4,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,-14.0,2018,1,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,-6.0,2018,8,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,-16.0,2017,2,2017-02


#### 1. Sales by Year


In [35]:
sales_by_year = sales.groupby('order_year').agg(
                    total_sales=('price', 'sum'),
                    total_freight=('freight_value', 'sum'),
                    total_items=('order_item_id', 'count'),
                    total_orders=('order_id', 'nunique')
                ).reset_index()

print("Sales By Year")
print("_"*60)
sales_by_year

Sales By Year
____________________________________________________________


,order_year,total_sales,total_freight,total_items,total_orders
0,2016,49785.92,7397.29,370,312
1,2017,6155806.98,986865.45,50864,44579
2,2018,7386050.80,1257646.80,61416,53775


#### 2. Sales by Month


In [36]:
sales_by_month = sales.groupby('order_month').agg(
                    total_sales=('price', 'sum'),
                    total_freight=('freight_value', 'sum'),
                    total_items=('order_item_id', 'count'),
                    total_orders=('order_id', 'nunique')
                ).reset_index()

print("Sales By Month")
print("_"*60)
sales_by_month

Sales By Month
____________________________________________________________


,order_month,total_sales,total_freight,total_items,total_orders
0,1,1070343.23,174147.15,9163,8009
1,2,1091481.73,181707.85,9623,8427
2,3,1357557.74,229617.67,11217,9829
3,4,1356574.98,215545.30,10659,9325
4,5,1502588.82,233383.95,12061,10513
5,6,1298162.91,227477.24,10661,9377
6,7,1393538.70,250160.95,11611,10242
7,8,1428658.01,242855.06,12158,10745
8,9,624814.05,96106.07,4838,4247
9,10,713727.09,112394.12,5685,4876


#### 3. Sales by Year & Month


In [37]:
sales_by_year_month = sales.groupby(['order_year', 'order_month']).agg(
                    total_sales=('price', 'sum'),
                    total_freight=('freight_value', 'sum'),
                    total_items=('order_item_id', 'count'),
                    total_orders=('order_id', 'nunique')
                ).reset_index().sort_values(['order_year', 'order_month'])

print("Sales By Year-Month")
print("_"*60)
sales_by_year_month

Sales By Year-Month
____________________________________________________________


,order_year,order_month,total_sales,total_freight,total_items,total_orders
0,2016,9,267.36,87.39,6,3
1,2016,10,49507.66,7301.18,363,308
2,2016,12,10.90,8.72,1,1
3,2017,1,120312.87,16875.62,955,789
4,2017,2,247303.02,38977.60,1951,1733
5,2017,3,374344.30,57704.29,3000,2641
6,2017,4,359927.23,52495.01,2684,2391
7,2017,5,506071.14,80119.81,4136,3660
8,2017,6,433038.60,69924.44,3583,3217
9,2017,7,498031.48,86940.14,4519,3969


### 5.2 Running Total & Average of Sales

In [41]:
# Cumulative Sum

monthly_sales = sales.groupby('order_year').agg(
    total_sales=('price', 'sum'),
    avg_sales=('price', 'mean') 
).reset_index().sort_values('order_year')

monthly_sales['running_total_sales'] = monthly_sales['total_sales'].cumsum()
monthly_sales['moving_average_sales'] = monthly_sales['avg_sales'].cumsum()

In [42]:
print(f"Running Total Sales\n{monthly_sales['running_total_sales']}")

print("_"*60)

print(f"Moving Average Sales\n{monthly_sales['moving_average_sales']}")

Running Total Sales
0       49785.92
1     6205592.90
2    13591643.70
Name: running_total_sales, dtype: float64
____________________________________________________________
Moving Average Sales
0    134.556541
1    255.581371
2    375.844019
Name: moving_average_sales, dtype: float64


### 5.3 Category Distribution

In [44]:
# order_items + products + category_translation
items_products = order_items.merge(
    products,
    on='product_id',
    how='left'
)

items_products = items_products.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

items_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english_x,product_category_name_english_y
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,58.0,598.0,4.0,650.0,28.0,9.0,14.0,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,56.0,239.0,2.0,30000.0,50.0,30.0,40.0,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,59.0,695.0,2.0,3050.0,33.0,13.0,33.0,furniture_decor,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,42.0,480.0,1.0,200.0,16.0,10.0,15.0,perfumery,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,59.0,409.0,1.0,3750.0,35.0,40.0,30.0,garden_tools,garden_tools


In [47]:
category_sales = items_products.groupby('product_category_name_english_x').agg(
    total_sales=('price', 'sum')
).reset_index().sort_values('total_sales', ascending=False)

category_sales.head(10)

,product_category_name_english_x,total_sales
43,health_beauty,1258681.34
70,watches_gifts,1205005.68
7,bed_bath_table,1036988.68
65,sports_leisure,988048.97
15,computers_accessories,911954.32
39,furniture_decor,729762.49
20,cool_stuff,635290.85
49,housewares,632248.66
5,auto,592720.11
42,garden_tools,485256.46


In [59]:
category_sales['overall_sales'] = category_sales['total_sales'].sum()
category_sales['contribution_percentage'] = (
    (category_sales['total_sales'] / category_sales['overall_sales']) * 100
).round(2)

category_sales = category_sales.sort_values('contribution_percentage', ascending=False)

print("Category Distribution")
print("_"*90)
category_sales.head(10)

Category Distribution
__________________________________________________________________________________________


,product_category_name_english_x,total_sales,overall_sales,contribution_precentage,contribution_percentage
43,health_beauty,1258681.34,13406593.94,9.388524,9.39
70,watches_gifts,1205005.68,13406593.94,8.988157,8.99
7,bed_bath_table,1036988.68,13406593.94,7.734915,7.73
65,sports_leisure,988048.97,13406593.94,7.369873,7.37
15,computers_accessories,911954.32,13406593.94,6.802282,6.80
39,furniture_decor,729762.49,13406593.94,5.443310,5.44
20,cool_stuff,635290.85,13406593.94,4.738645,4.74
49,housewares,632248.66,13406593.94,4.715953,4.72
5,auto,592720.11,13406593.94,4.421109,4.42
42,garden_tools,485256.46,13406593.94,3.619536,3.62


### 5.4 Delivery Performance
- Measure Delivery Delay
- Percentage of Late Deliveries

In [60]:
orders['delivery_delay'] = (orders['order_delivered_customer_date'] - orders['order_estimated_delivery_date']).dt.days
# If Negative -> Delivered Early
# If Positive -> Delivered Late
# If Zero     -> Delivered On Time

orders.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,2017,10,2017-10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,2018,7,2018-07
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,2018,8,2018-08


In [61]:
late_deliveries = (orders['delivery_delay'] > 0).mean() * 100
print(f"Percentage of late deliveries: {late_deliveries:.2f}%")

Percentage of late deliveries: 6.57%


In [62]:
# Average of Late Delivery Days
mean_late_delivery_days = orders.loc[orders['delivery_delay'] > 0, 'delivery_delay'].mean()
print(f"Average of Late Delivery Days = {mean_late_delivery_days:.2f}")

Average of Late Delivery Days = 10.62


#### Relationship between Review Score and Late Delivery

In [63]:
orders_order_reviews = orders.merge(order_reviews, on='order_id')
orders_order_reviews.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,2017,10,2017-10,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,2018,7,2018-07,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,2018,8,2018-08,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58


In [64]:
# Make Delivery Status
orders_order_reviews['delivery_status'] = orders_order_reviews['delivery_delay'].apply(
    lambda x: 'Late' if x > 0 else 'Early' if x < 0 else 'On Time'
)

orders_order_reviews.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,delivery_status
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,2017,10,2017-10,a54f0611adc9ed256b57ede6b6eb5114,4,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11 00:00:00,2017-10-12 03:43:48,Early
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,2018,7,2018-07,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08 00:00:00,2018-08-08 18:37:50,Early
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,2018,8,2018-08,e73b67b67587f7644d5bd1a52deb1b01,5,NaN,NaN,2018-08-18 00:00:00,2018-08-22 19:07:58,Early


In [65]:
# Average Review Score based on the Delivery Status
avg_score_by_status = orders_order_reviews.groupby('delivery_status')['review_score'].mean().round(2)
avg_score_by_status

delivery_status
Early      4.29
Late       2.27
On Time    2.47
Name: review_score, dtype: float64

### 5.5 Customer Satisfaction

In [66]:
order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53


**Review Scores Distribution**

In [67]:
order_reviews['review_score'].value_counts()

review_score
5    57328
4    19142
1    11424
3     8179
2     3151
Name: count, dtype: int64

**What are the Best and Worst Categories based on the review score ?**

In [68]:
prod_items = products.merge(order_items, on='product_id', how='inner')

prod_items.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,order_id,order_item_id,seller_id,shipping_limit_date,price,freight_value
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,e17e4f88e31525f7deef66779844ddce,1,5670f4db5b62c43d542e1b2d56b0cf7c,2018-04-30 17:33:54,10.91,7.39
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,5236307716393b7114b53ee991f36956,1,b561927807645834b59ef0d16ba55a24,2018-02-06 19:11:15,248.00,17.99
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,01f66e58769f84129811d43eefd187fb,1,7b07b3c7487f0ea825fc6df75abd658b,2018-07-11 21:30:20,79.80,7.82


In [69]:
prod_items_reviews = prod_items.merge(order_reviews, on='order_id', how='inner')

prod_items_reviews.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,...,seller_id,shipping_limit_date,price,freight_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,...,5670f4db5b62c43d542e1b2d56b0cf7c,2018-04-30 17:33:54,10.91,7.39,d71da8fd8c6e3adef26be965f065b8a1,5,Vale a pena,Super adooorei o delineador ele é bem preto e ...,2018-04-28 00:00:00,2018-05-04 00:55:02
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,...,b561927807645834b59ef0d16ba55a24,2018-02-06 19:11:15,248.00,17.99,19866247db89845ef558ef19a6c26fcd,5,NaN,Excelente apontador! Em 3 segundos uma ponta p...,2018-02-17 00:00:00,2018-02-18 23:24:14
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,...,7b07b3c7487f0ea825fc6df75abd658b,2018-07-11 21:30:20,79.80,7.82,32921b4e09c9a19d61fe46ce04231b84,5,NaN,NaN,2018-07-14 00:00:00,2018-07-16 16:20:04


In [70]:
prod_items_reviews = prod_items_reviews.merge(
    category_translation,
    on='product_category_name',
    how='left'
)

prod_items_reviews.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english_x,...,shipping_limit_date,price,freight_value,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,product_category_name_english_y
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,...,2018-04-30 17:33:54,10.91,7.39,d71da8fd8c6e3adef26be965f065b8a1,5,Vale a pena,Super adooorei o delineador ele é bem preto e ...,2018-04-28 00:00:00,2018-05-04 00:55:02,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,...,2018-02-06 19:11:15,248.00,17.99,19866247db89845ef558ef19a6c26fcd,5,NaN,Excelente apontador! Em 3 segundos uma ponta p...,2018-02-17 00:00:00,2018-02-18 23:24:14,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,...,2018-07-11 21:30:20,79.80,7.82,32921b4e09c9a19d61fe46ce04231b84,5,NaN,NaN,2018-07-14 00:00:00,2018-07-16 16:20:04,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,...,2018-08-07 09:10:13,112.30,9.54,70949d129965fb1ee32aaccec720a19e,1,produto divergente,modelo comprado: travesseiro memoria carinho B...,2018-08-07 00:00:00,2018-08-08 11:53:04,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,...,2018-04-17 01:30:23,37.90,8.29,74c9c7d0bc153bbbc92e4c1597b8ba55,5,NaN,NaN,2018-04-17 00:00:00,2018-04-18 04:29:47,housewares


In [71]:
prod_items_reviews.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english_x', 'order_id', 'order_item_id',
       'seller_id', 'shipping_limit_date', 'price', 'freight_value',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'product_category_name_english_y'],
      dtype='object')

In [76]:
category_scores = (
    prod_items_reviews.groupby('product_category_name_english_x')['review_score']
    .agg(['mean', 'count'])
    .reset_index()
)

category_scores.head()

,product_category_name_english_x,mean,count
0,agro_industry_and_commerce,4.000000,212
1,air_conditioning,3.969178,292
2,art,3.937198,207
3,arts_and_craftmanship,4.125000,24
4,audio,3.825485,361


In [77]:
# Worst Products that has lowest mean score
worst_categories = category_scores.sort_values('mean', ascending=True).head(10)

worst_categories

,product_category_name_english_x,mean,count
61,security_and_services,2.500000,2
23,diapers_and_hygiene,3.256410,39
57,office_furniture,3.493183,1687
46,home_comfort_2,3.629630,27
30,fashion_male_clothing,3.641221,131
34,fixed_telephony,3.683206,262
58,party_supplies,3.767442,43
27,fashio_female_clothing,3.780000,50
41,furniture_mattress_and_upholstery,3.815789,38
4,audio,3.825485,361


In [78]:
# Top 10 Products based on Reviews
best_categories = category_scores.sort_values('mean', ascending=False).head(10)
best_categories

,product_category_name_english_x,mean,count
11,cds_dvds_musicals,4.642857,14
29,fashion_childrens_clothes,4.500000,8
8,books_general_interest,4.446266,549
22,costruction_tools_tools,4.444444,99
35,flowers,4.419355,31
9,books_imported,4.400000,60
10,books_technical,4.368421,266
37,food_drink,4.315412,279
53,luggage_accessories,4.315257,1088
64,small_appliances_home_oven_and_coffee,4.302632,76


### 5.5 Customer Segmentation (VIP/ Regular/ New)

In [80]:
# Customer Spending + Lifespan
sales_customers = order_items.merge(
    orders,
    on='order_id',
    how='inner'
)

sales_customers.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,-9.0,2017,9,2017-09
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,-3.0,2017,4,2017-04
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,-14.0,2018,1,2018-01
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,-6.0,2018,8,2018-08
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,-16.0,2017,2,2017-02


In [81]:
sales_customers = sales_customers.merge(
    customers,
    on='customer_id',
    how='left'
)

sales_customers.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,customer_id,order_status,order_purchase_timestamp,...,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,...,2017-09-20 23:43:48,2017-09-29,-9.0,2017,9,2017-09,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,...,2017-05-12 16:04:24,2017-05-15,-3.0,2017,4,2017-04,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,...,2018-01-22 13:19:16,2018-02-05,-14.0,2018,1,2018-01,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,...,2018-08-14 13:32:39,2018-08-20,-6.0,2018,8,2018-08,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,...,2017-03-01 16:42:31,2017-03-17,-16.0,2017,2,2017-02,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP


In [82]:
customer_spending = sales_customers.groupby('customer_unique_id').agg(
    total_spending=('price', 'sum'),
        first_order=('order_purchase_timestamp', 'min'),
        last_order=('order_purchase_timestamp', 'max'),
        total_orders=('order_id', 'nunique')
    ).reset_index()

customer_spending['lifespan_days'] = (customer_spending['last_order'] - customer_spending['first_order']).dt.days

customer_spending['lifespan_months'] = customer_spending['lifespan_days'] / 30

customer_spending.head()

,customer_unique_id,total_spending,first_order,last_order,total_orders,lifespan_days,lifespan_months
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,2018-05-10 10:56:27,2018-05-10 10:56:27,1,0,0.0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,2018-05-07 11:11:27,2018-05-07 11:11:27,1,0,0.0
2,0000f46a3911fa3c0805444483337064,69.00,2017-03-10 21:05:03,2017-03-10 21:05:03,1,0,0.0
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,2017-10-12 20:29:41,2017-10-12 20:29:41,1,0,0.0
4,0004aac84e0df4da2b147fca70cf8255,180.00,2017-11-14 19:45:42,2017-11-14 19:45:42,1,0,0.0


In [83]:
def segment_customer(row):
    if row['lifespan_months'] >= 6 and row['total_spending'] > 500:
        return 'VIP'
    elif row['lifespan_months'] >= 6 and row['total_spending'] <= 500:
        return 'Regular'
    else:
        return 'New'

In [84]:
customer_spending['customer_segment'] = customer_spending.apply(segment_customer, axis=1)
customer_spending.head()

,customer_unique_id,total_spending,first_order,last_order,total_orders,lifespan_days,lifespan_months,customer_segment
0,0000366f3b9a7992bf8c76cfdf3221e2,129.90,2018-05-10 10:56:27,2018-05-10 10:56:27,1,0,0.0,New
1,0000b849f77a49e4a4ce2b2a4ca5be3f,18.90,2018-05-07 11:11:27,2018-05-07 11:11:27,1,0,0.0,New
2,0000f46a3911fa3c0805444483337064,69.00,2017-03-10 21:05:03,2017-03-10 21:05:03,1,0,0.0,New
3,0000f6ccb0745a6a4b88665a16c9f078,25.99,2017-10-12 20:29:41,2017-10-12 20:29:41,1,0,0.0,New
4,0004aac84e0df4da2b147fca70cf8255,180.00,2017-11-14 19:45:42,2017-11-14 19:45:42,1,0,0.0,New


In [85]:
segment_summary = (
    customer_spending.groupby('customer_segment')
    .agg(
        total_customers=('customer_unique_id', 'count'),
        avg_spending=('total_spending', 'mean'),
        avg_orders=('total_orders', 'mean')
    )
    .reset_index()
)


print("Customer Segments")
print("_"*60)
segment_summary

Customer Segments
____________________________________________________________


,customer_segment,total_customers,avg_spending,avg_orders
0,New,94862,141.668704,1.026744
1,Regular,491,203.214114,2.158859
2,VIP,67,789.387910,3.089552


### 5.6 Sales by State / City & Customers

In [86]:
# Merge customers with orders
cust_orders = orders.merge(customers, on='customer_id')

cust_orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay,order_year,order_month,order_year_month,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-8.0,2017,10,2017-10,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-6.0,2018,7,2018-07,af07308b275d755c9edb36a90c618231,47813,barreiras,BA
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-18.0,2018,8,2018-08,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-13.0,2017,11,2017-11,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-10.0,2018,2,2018-02,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP


In [87]:
top_cities = customers['customer_state'].value_counts().head(5)

print(f"Top 5 Cities: \n{top_cities}")

Top 5 Cities: 
customer_state
SP    41746
RJ    12852
MG    11635
RS     5466
PR     5045
Name: count, dtype: int64


### 5.7 Payment Methods

In [88]:
order_payments['payment_type'].value_counts()

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

### 5.8 Top Product Categories

In [89]:
# Merge products with translation
products = products.merge(category_translation, on='product_category_name', how='left')

products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english_x,product_category_name_english_y
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,housewares
...,...,...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0,furniture_decor,furniture_decor
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0,construction_tools_lights,construction_tools_lights
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0,bed_bath_table,bed_bath_table
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0,computers_accessories,computers_accessories


In [90]:
# Then merge with order_items to get sales volume
prod_oredr_items = products.merge(order_items, on='product_id')
prod_oredr_items.head(3)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english_x,product_category_name_english_y,order_id,order_item_id,seller_id,shipping_limit_date,price,freight_value
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,perfumery,e17e4f88e31525f7deef66779844ddce,1,5670f4db5b62c43d542e1b2d56b0cf7c,2018-04-30 17:33:54,10.91,7.39
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,art,5236307716393b7114b53ee991f36956,1,b561927807645834b59ef0d16ba55a24,2018-02-06 19:11:15,248.00,17.99
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,sports_leisure,01f66e58769f84129811d43eefd187fb,1,7b07b3c7487f0ea825fc6df75abd658b,2018-07-11 21:30:20,79.80,7.82


### Machine Learning Systems Examples
- Customer Churn
- Delivery Days Prediction
- Review Score Prediction
- Time Series Forecasting
- Sentiment Analysis